In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression,Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
import category_encoders as ce

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

from sklearn.decomposition import PCA


In [3]:
import warnings
warnings.filterwarnings('ignore')

In [4]:
df = pd.read_csv(r"C:\Users\puruc\Downloads\House Price Prediction\STEP - 3 (Feature Engg)\gurgaon_properties_post_feature_selection.csv").drop(columns=['floor_category','balcony'])

In [5]:
df.head()

,property_type,sector,price,bedRoom,bathroom,agePossession,super_built_up_area,servant room,furnishing_type,luxury_category
0,flat,sector 47,2.40,3,3,Moderately Old,2992.86,No,semifurnished,Low
1,flat,sector 65,2.36,3,3,Relatively New,1650.00,Yes,semifurnished,Medium
2,flat,sector 111,2.25,3,3,Relatively New,1700.05,No,unfurnished,Medium
3,flat,sector 71,0.60,2,2,Relatively New,599.98,No,unfurnished,Medium
4,flat,sector 2,1.40,3,3,Moderately Old,1780.03,No,unfurnished,Medium


In [6]:
# Numerical = bedRoom, bathroom, super_built_up_area, servant room
# Ordinal = property_type, furnishing_type, luxury_category 
# OHE = sector, agePossession

#### `1.` Property Type

In [7]:
df['property_type'] = df['property_type'].replace({'flat':0,'house':1})

#### `2.` Age Possession

In [8]:
df['agePossession'] = df['agePossession'].replace(
    {
        'Relatively New':'new',
        'Moderately Old':'old',
        'New Property' : 'new',
        'Old Property' : 'old',
        'Under Construction' : 'under construction'
    }
)

#### `3.` Servant Room

In [9]:
df['servant room'] = df['servant room'].replace({'No':0,'Yes':1})

#### `4.` Furnishing Type

In [10]:
df['furnishing_type'] = df['luxury_category'].replace({'Low':0,'Medium':1,'High':2})

#### `5.` Luxury Category

In [11]:
df['luxury_category'] = df['luxury_category'].replace({'Low':0,'Medium':1,'High':2})

In [12]:
df.head()

,property_type,sector,price,bedRoom,bathroom,agePossession,super_built_up_area,servant room,furnishing_type,luxury_category
0,0,sector 47,2.40,3,3,old,2992.86,0,0,0
1,0,sector 65,2.36,3,3,new,1650.00,1,1,1
2,0,sector 111,2.25,3,3,new,1700.05,0,1,1
3,0,sector 71,0.60,2,2,new,599.98,0,1,1
4,0,sector 2,1.40,3,3,old,1780.03,0,1,1


#### `6` Age Possesion and sector

In [13]:
new_df = pd.get_dummies(df,columns=['sector','agePossession'],drop_first=True)

In [14]:
X = new_df.drop(columns=['price'])
y = new_df['price']

In [15]:
y_log = np.log1p(y)

In [17]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [18]:
X_scaled = pd.DataFrame(X_scaled,columns=X.columns)

In [19]:
X_scaled

,property_type,bedRoom,bathroom,super_built_up_area,servant room,furnishing_type,luxury_category,sector_gwal pahari,sector_manesar,sector_sector 1,...,sector_sector 92,sector_sector 93,sector_sector 95,sector_sector 99,sector_sector 99a,sector_sector 9a,sector_sohna road,sector_sohna road road,agePossession_old,agePossession_under construction
0,-0.505246,-0.125574,-0.171406,0.794932,-0.742617,-0.984739,-0.984739,-0.071786,-0.097407,-0.041374,...,-0.170339,-0.050695,-0.126152,-0.056061,-0.092834,-0.053445,-0.212724,-0.050695,1.676950,-0.29094
1,-0.505246,-0.125574,-0.171406,-0.325787,1.346589,0.437482,0.437482,-0.071786,-0.097407,-0.041374,...,-0.170339,-0.050695,-0.126152,-0.056061,-0.092834,-0.053445,-0.212724,-0.050695,-0.596321,-0.29094
2,-0.505246,-0.125574,-0.171406,-0.284017,-0.742617,0.437482,0.437482,-0.071786,-0.097407,-0.041374,...,-0.170339,-0.050695,-0.126152,-0.056061,-0.092834,-0.053445,-0.212724,-0.050695,-0.596321,-0.29094
3,-0.505246,-0.870318,-0.880246,-1.202109,-0.742617,0.437482,0.437482,-0.071786,-0.097407,-0.041374,...,-0.170339,-0.050695,-0.126152,-0.056061,-0.092834,-0.053445,-0.212724,-0.050695,-0.596321,-0.29094
4,-0.505246,-0.125574,-0.171406,-0.217267,-0.742617,0.437482,0.437482,-0.071786,-0.097407,-0.041374,...,-0.170339,-0.050695,-0.126152,-0.056061,-0.092834,-0.053445,-0.212724,-0.050695,1.676950,-0.29094
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3506,-0.505246,-0.870318,-0.880246,-0.308269,-0.742617,-0.984739,-0.984739,-0.071786,-0.097407,-0.041374,...,-0.170339,-0.050695,-0.126152,-0.056061,-0.092834,-0.053445,-0.212724,-0.050695,-0.596321,-0.29094
3507,-0.505246,-0.125574,-0.171406,-0.006161,-0.742617,-0.984739,-0.984739,-0.071786,-0.097407,-0.041374,...,-0.170339,-0.050695,-0.126152,-0.056061,-0.092834,-0.053445,-0.212724,-0.050695,-0.596321,-0.29094
3508,-0.505246,-0.870318,-0.880246,-0.334141,-0.742617,-0.984739,-0.984739,-0.071786,-0.097407,-0.041374,...,-0.170339,-0.050695,-0.126152,-0.056061,-0.092834,-0.053445,-0.212724,-0.050695,-0.596321,-0.29094
3509,-0.505246,0.619171,0.537434,0.125710,-0.742617,0.437482,0.437482,-0.071786,-0.097407,-0.041374,...,-0.170339,-0.050695,-0.126152,-0.056061,-0.092834,-0.053445,-0.212724,-0.050695,-0.596321,-0.29094


In [20]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(LinearRegression(), X_scaled, y_log, cv=kfold, scoring='r2')

In [21]:
scores.mean(),scores.std()

(0.8523569949240647, 0.01156771860302483)

In [22]:
lr = LinearRegression()
ridge = Ridge(alpha=0.0001)

In [23]:
lr.fit(X_scaled,y_log)

LinearRegression()

In [24]:
ridge.fit(X_scaled,y_log)

Ridge(alpha=0.0001)

In [26]:
lr.coef_.shape

(119,)

In [27]:
coef_df = pd.DataFrame(ridge.coef_.reshape(1,119),columns=X.columns).stack().reset_index().drop(columns=['level_0']).rename(columns={'level_1':'feature',0:'coef'})

In [28]:
coef_df

,feature,coef
0,property_type,0.112110
1,bedRoom,0.013193
2,bathroom,0.074651
3,super_built_up_area,0.232203
4,servant room,0.043067
...,...,...
114,sector_sector 9a,-0.003453
115,sector_sohna road,-0.023850
116,sector_sohna road road,-0.010273
117,agePossession_old,-0.014783


In [29]:
# 1. Import necessary libraries
import statsmodels.api as sm

# 2. Add a constant to X
X_with_const = sm.add_constant(X_scaled)

# 3. Fit the model
model = sm.OLS(y_log, X_with_const).fit()

# 4. Obtain summary statistics
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                  price   R-squared:                       0.866
Model:                            OLS   Adj. R-squared:                  0.861
Method:                 Least Squares   F-statistic:                     186.0
Date:                Wed, 25 Sep 2024   Prob (F-statistic):               0.00
Time:                        14:49:48   Log-Likelihood:                 688.20
No. Observations:                3511   AIC:                            -1138.
Df Residuals:                    3392   BIC:                            -404.9
Df Model:                         118                                         
Covariance Type:            nonrobust                                         
                                        coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
const 

stand_coef = unstand_coef * {(std x)/(std y)}

In [39]:
stand_coef = 0.0132
std_x = X['bedRoom'].std()
std_y = y_log.std()

In [40]:
unstand_coef = (stand_coef * std_y)/std_x

In [41]:
unstand_coef

0.005344439938446593

In [43]:
np.expm1(0.005344439938446593)

0.005358746933844827